# Control de un player de vídeo a través de gestos

## Paquetes necesarios

In [2]:
import cv2
import mediapipe as mp
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
from selenium.webdriver.support.ui import WebDriverWait
import pyautogui
import math
import numpy as np
from selenium.webdriver.support.ui import WebDriverWait

### Clase para controlar el servidor

In [3]:
class VideoController:
    def __init__(self, server_path):
        options = Options()
        options.add_argument("--autoplay-policy=no-user-gesture-required")
        options.add_argument("--disable-features=FullscreenBlocking")
        self.driver = webdriver.Chrome(options=options)
        self.driver.get(server_path)
        self.server_path = self.driver.current_url  
    def click(self):
        pyautogui.click()

        self.driver.execute_script("video = document.getElementById('video');")
        WebDriverWait(self.driver, 10).until(
            lambda d: d.execute_script("if (video) {return video.readyState};") == 4  #Espera a que cargue el elemento video en la web
        )
        current_url = self.driver.current_url
        return current_url

    def adelantar(self):
        print("adelantar 5s")
        self.driver.execute_script("video.currentTime += 5;")

    def atrasar(self):
        print("atrasar 5s")
        self.driver.execute_script("video.currentTime -= 5;")

    def subir_volumen(self):
        print("subir volumen")
        self.driver.execute_script("if (video.volume <= 0.9){video.volume += 0.1}else {video.volume=1};")

    def bajar_volumen(self):
        print("bajar_volumen")
        self.driver.execute_script("if (video.volume >= 0.1){video.volume -= 0.1}else {video.volume=0};")

    def pause(self):
        print("pausar")
        self.driver.execute_script("video.pause();")

    def play(self):
        print("reproducir")
        self.driver.execute_script("video.play();")

    def menu(self):
        print("salir al menú")
        self.driver.get(self.server_path)

### Funciones de detección dedos y gestos

In [4]:
def get_finger_states(hand_landmarks):
    fingers = []
    if hand_landmarks.landmark[4].x > hand_landmarks.landmark[3].x:
        fingers.append(1)
    else:
        fingers.append(0)
    tips = [8, 12, 16, 20]
    pips = [6, 10, 14, 18]
    for tip, pip in zip(tips, pips):
        fingers.append(1 if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[pip].y else 0)
    return fingers # [pulgar, índice, corazón, anular, meñique]

In [14]:
def detect_custom_gesture(fingers):
    # fingers = [pulgar, índice, medio, anular, meñique]
    if fingers == [1, 1, 1, 0, 0]:
        return "3"
    #elif fingers == [0, 1, 1, 0, 0]:
     #   return "V"
    if fingers == [0, 1, 0, 0, 1]:
        return "Rock"
    #elif fingers == [0, 0, 0, 0, 0]:
     #   return "Puño"
    if fingers == [1, 1, 1, 1, 1]:
        return "Palma"
    if fingers == [1, 0, 0, 0, 1]:
        return "modo_pose"
    else:
        return "Unknown"

### Función de control

In [79]:
def control(modo, controller, pos_anterior, punta_indice, gesture):
    global index_url
    global previousMilis
    currentMilis = time.time()
    screen_w, screen_h = pyautogui.size() # Dimensiones de la pantalla en píxeles
        
    if modo == "navegación":
        if pos_anterior:
            if punta_indice.y - pos_anterior[8].y > 0.08:
                print('click')
                current_url = controller.click()
                if index_url != current_url:
                    return "player"
                    
            # Para que no haga el click y el movimiento a la vez
            else: 
                pyautogui.moveTo(int(punta_indice.x * screen_w), int(punta_indice.y * screen_h))
        return "navegación"
    
    else:  # modo player
        if pos_anterior and currentMilis - previousMilis > 0.5:
            dx = punta_indice.x - pos_anterior[8].x
            dy = punta_indice.y - pos_anterior[8].y

            if dx > 0.04:
                controller.adelantar()
                gesture = 'der-izq'
            elif dx < -0.04:
                controller.atrasar()
                gesture = 'izq-der'
            if dy < -0.08:
                controller.subir_volumen()
                gesture = 'abajo-arriba'
            elif dy > 0.08:
                controller.bajar_volumen()
                gesture = 'arriba-abajo'

            if gesture != 'Unknown':
                print('cooldown')
                previousMilis = currentMilis
            
            if gesture == "Palma":
                controller.pause()
            elif gesture == "Rock":
                controller.play()
            #elif gesture == "V":
             #   controller.cambiar_pantalla()
            elif gesture == "3":
                controller.menu()
                return "navegación"
    return "player"

### Función main

In [80]:

#index_url = "C:/Users/34611/OneDrive/Escritorio/GTDM/Tercero/Stream/Practica6/index.html"
index_url = 'https://areyousure.wavedash.com'
controller = VideoController(index_url)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
cap = cv2.VideoCapture(0) # Probar a cambiar entre 1 o 0

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)
img_bgr = cv2.imread('surprise.jpg')
img_landmarks = pose_nueva(img_bgr)
img_bgr = cv2.resize(img_bgr,(640,480))
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

modo = "navegación"
pos_anterior = None
previousMilis = 0

with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("No se puede leer la cámara")
            break
        
        try:
            if modo == 'pose':
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
                # Procesar con MediaPipe Pose
                result = pose.process(frame_rgb)
            
                # Dibujar resultados
               
                if result.pose_landmarks:
                    landmarks_centrados = centrar_landmarks(result.pose_landmarks.landmark)
                    if comparar_poses(landmarks_centrados, img_landmarks,0.4):
                        #cv2.putText(frame, f'Pose: {'Sorpresa'}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                        cv2.imshow('Pose en Vivo',mezclar(img_bgr,frame))
                    else:
                        #mp_drawing.draw_landmarks(frame, result.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                        cv2.imshow('Pose en Vivo', frame)
            else:
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image.flags.writeable = False
                results = hands.process(image)
                image.flags.writeable = True
                image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
                if results.multi_hand_landmarks:
                    for hand_landmarks in results.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                        
                        punta_indice = hand_landmarks.landmark[8]
                        dedos        = get_finger_states(hand_landmarks)
                        gesture      = detect_custom_gesture(dedos)
                        if gesture == 'modo_pose':
                            modo = 'pose'
                            continue
                        modo = control(modo, controller, pos_anterior, punta_indice, gesture)
                        pos_anterior        = hand_landmarks.landmark
    
                        cv2.putText(image, f'Gesto: {gesture}', (10, 30),cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                cv2.imshow("Control con la mano", image)

        except Exception as e:
            import traceback
            traceback.print_exc()

        if cv2.waitKey(1) == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

click
atrasar 5s
cooldown
cooldown
salir al menú
click
subir volumen
cooldown
cooldown
pausar
cooldown
pausar
cooldown
pausar
bajar_volumen
cooldown
subir volumen
cooldown
bajar_volumen
cooldown
bajar_volumen
cooldown
bajar_volumen
cooldown
bajar_volumen
cooldown
bajar_volumen
cooldown
bajar_volumen
cooldown
bajar_volumen
cooldown
bajar_volumen
cooldown
subir volumen
cooldown
subir volumen
cooldown
subir volumen
cooldown
subir volumen
cooldown
subir volumen
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
atrasar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
atrasar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
atrasar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
atrasar 5s
cooldown
atrasar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
adelantar 5s
cooldown
atrasar 5s
cooldown
adelantar 5s


## A partir de aquí poses

In [56]:
def distancia_euclidiana(l1, l2):
    return math.sqrt((l1[0] - l2[0])**2 + (l1[1] - l2[1])**2 + (l1[2] - l2[2])**2)

def comparar_poses(pose_actual, pose_referencia, umbral=0.4):
    total_dist = 0
    num_puntos = len(pose_actual)

    for i in range(num_puntos):
        dist = distancia_euclidiana(pose_actual[i], pose_referencia[i])
        total_dist += dist

    promedio = total_dist / num_puntos
    #print(f"Distancia promedio: {promedio:.4f}")
    return promedio < umbral

In [10]:
def centrar_landmarks(landmarks):
    cadera_izq = landmarks[23]
    cadera_der = landmarks[24]
    x = (cadera_izq.x + cadera_der.x) / 2
    y = (cadera_izq.y + cadera_der.y) / 2
    z = (cadera_izq.z + cadera_der.z) / 2
    landmarks_centrados = []
    for l in landmarks:
        l_centrado = [0,0,0]
        l_centrado[0] = l.x - x
        l_centrado[1] = l.y - y
        l_centrado[2] = l.z - z
        landmarks_centrados.append(l_centrado)
    return landmarks_centrados

In [49]:
def mezclar(img,frame):
    img = img.astype(np.float32) / 255.0
    frame = frame.astype(np.float32) / 255.0
    # Nivel de opacidad (alpha): 0 = totalmente transparente, 1 = totalmente opaca
    alpha = 0.5
    
    # Mezclar imagen con el fondo
    img_con_opacidad = cv2.addWeighted(frame, alpha, img, 1 - alpha, 0)
    
    # Volver a tipo uint8 para mostrar
    return (img_con_opacidad * 255).astype(np.uint8)

In [45]:
def pose_nueva(img_bgr):
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Procesar imagen
    resultado = pose.process(img_rgb)
    
    # Dibujar resultados si hay landmarks
    if resultado.pose_landmarks:
        # Mostrar imagen con puntos dibujados
        img_landmarks = centrar_landmarks(resultado.pose_landmarks.landmark)
        return img_landmarks
    else:
        print("No se detectó ninguna persona en la imagen.")

In [42]:
def modo_pose():
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=True)
    mp_drawing = mp.solutions.drawing_utils

    img_bgr = cv2.imread('surprise.jpg')
    img_landmarks = pose_nueva(img_bgr)
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
    
        # Convertir a RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
        # Procesar con MediaPipe Pose
        result = pose.process(frame_rgb)
    
        # Dibujar resultados
        try:
            if result.pose_landmarks:
                landmarks_centrados = centrar_landmarks(result.pose_landmarks.landmark)
                if comparar_poses(landmarks_centrados, img_landmarks,0.4):
                    #cv2.putText(frame, f'Pose: {'Sorpresa'}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                    cv2.imshow('Pose en Vivo',img_bgr)
                else:
                    #mp_drawing.draw_landmarks(frame, result.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                    cv2.imshow('Pose en Vivo', frame)
        except Exception as e:
                import traceback
                traceback.print_exc()
    
        # Salir con 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Liberar recursos
    cap.release()
    cv2.destroyAllWindows()